# 05 — Conclusiones

Este notebook cierra el proyecto respondiendo a las preguntas de análisis definidas al
inicio, diferenciando explícitamente **evidencia**, **interpretación** y **conclusión**
para cada una, y coherente con lo desarrollado en `03_eda.ipynb` y `04_pca.ipynb`.


## 0. Entorno de ejecución (Google Colab / local)

Si corre en Colab, monta Google Drive y usa `BASE_PATH` apuntando a la carpeta del
proyecto en tu Drive (subí `PI_Mineria_Datos_1/` completa). Si corre localmente, usa la
ruta relativa habitual.


In [1]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BASE_PATH = "/content/drive/MyDrive/PI_Mineria_Datos_1" if IN_COLAB else ".."

print(f"Ejecutando en Colab: {IN_COLAB}")
print(f"BASE_PATH: {BASE_PATH}")


Ejecutando en Colab: False
BASE_PATH: ..


In [2]:
import pandas as pd
df = pd.read_csv(f"{BASE_PATH}/data/processed/streaming_users_clean.csv")
log = pd.read_csv(f"{BASE_PATH}/logs/pipeline_log.csv")
print(f"Dataset final: {df.shape[0]} usuarios, {df.shape[1]} columnas")
log


Dataset final: 8000 usuarios, 8 columnas


## Pregunta 1 — ¿Cómo se distribuyen la edad y el consumo mensual?

- **Evidencia:** histogramas de `age` (unimodal, 25-45 años concentra la mayoría) y de
  `monthly_watch_time_mins` (asimétrica a la derecha, mediana ≈ 690 min).
- **Interpretación:** la base de usuarios es mayoritariamente adulta joven/media, con un
  grupo minoritario de consumo muy alto que estira la distribución.
- **Conclusión:** para reportar el consumo "típico" de la plataforma conviene usar la
  mediana y no el promedio, dado el sesgo de la distribución.

## Pregunta 2 — ¿El plan de suscripción se relaciona con consumo o tickets de soporte?

- **Evidencia:** medianas de consumo crecientes por plan (Básico 557.6 min, Estándar
  832.6 min, Premium 1080.9 min); promedio de tickets de soporte similar entre planes.
- **Interpretación:** existe una asociación agregada entre plan y volumen de consumo
  (aunque con alta dispersión individual dentro de cada plan), pero no entre plan y
  carga de soporte.
- **Conclusión:** el plan de suscripción es un buen proxy del nivel de consumo esperado
  a nivel agregado, pero no explica por sí solo el comportamiento de un usuario
  individual, ni predice su necesidad de soporte.

## Pregunta 3 — ¿Existen diferencias de consumo según país o género favorito?

- **Evidencia:** el consumo promedio por país varía en un rango acotado (~781 a ~807
  minutos entre los 7 países), sin un país que se destaque.
- **Interpretación:** el país de origen no parece ser un factor relevante de
  segmentación de consumo en este dataset.
- **Conclusión:** las estrategias de producto o contenido no deberían priorizarse por
  país en base únicamente a esta variable de consumo agregado.

## Pregunta 4 — ¿Cómo interactúan edad, consumo y tickets, diferenciados por plan?

- **Evidencia:** el scatter edad–consumo por plan no muestra separación por color; la
  proyección PCA (PC1–PC2) tampoco muestra agrupamientos por plan; las tres variables
  numéricas tienen correlación cercana a 0 entre sí, y PCA reparte la varianza casi por
  igual entre sus 3 componentes (~33% cada una).
- **Interpretación:** no existe una combinación lineal simple de estas tres variables
  que separe a los usuarios por plan de suscripción; la asociación plan–consumo hallada
  en la pregunta 2 no está mediada por la edad ni acompañada de una señal similar en los
  tickets de soporte.
- **Conclusión:** con las variables numéricas disponibles, **no es posible ni conveniente
  reducir la dimensionalidad** del dataset sin perder información relevante; para
  explicar mejor el plan de suscripción sería necesario incorporar variables adicionales
  no disponibles en este dataset (por ejemplo, antigüedad como usuario, dispositivo de
  uso, o número de perfiles en la cuenta).


## Limitaciones

- La imputación por mediana en `age`, `monthly_watch_time_mins` y
  `customer_support_tickets` (aplicada a 120, 273 y 96 registros respectivamente,
  según `logs/pipeline_log.csv`) reduce la varianza real de esas variables y puede
  atenuar relaciones que existieran en los datos originales sin errores.
- 462 registros (~5.8% del dataset final) quedaron sin `last_login_date` válida, por lo
  que cualquier análisis de recencia de uso debe interpretarse sobre una base parcial.
- Para los `user_id` duplicados con valores distintos entre sí se conservó el primer
  registro observado, sin evidencia disponible en el dataset para determinar cuál de los
  registros era el correcto.
- El alcance de las conclusiones se encuentra condicionado por la información disponible
  y por las decisiones documentadas durante el proceso.

## Mejoras futuras

- Una mejora futura podría consistir en incorporar información adicional (antigüedad de
  la cuenta, dispositivo de reproducción, cantidad de perfiles) que permita ampliar el
  alcance del análisis y explicar mejor la asociación entre plan de suscripción y
  comportamiento de uso.
- Podría evaluarse un PCA que incluya variables categóricas codificadas (plan, país,
  género favorito) para verificar si aportan estructura adicional que las variables
  puramente numéricas no capturan.
- Sería valioso contar con una fecha de alta del usuario, además de la de último login,
  para poder estimar antigüedad y analizar retención en el tiempo.
